# 05. SQLによるデータ管理・分析

## 目的

これまでPython（pandas）で処理してきた日次集計データを
SQLiteデータベースに格納し、SQLによる集計・比較を行う。

SQLを用いてTrial別・Pen別の傾向を確認するとともに、
ウィンドウ関数を活用した時系列分析を実施する。

PythonとSQLを組み合わせることで、
データの保存・抽出・分析を行う一連の処理を構築する。

In [1]:
from pathlib import Path
import pandas as pd
import sqlite3

PROJECT_ROOT = Path("..")

trial1 = pd.read_csv(
    PROJECT_ROOT / "data/interim/trial1_daily_quality_summary.csv",
    parse_dates=["date"]
)

trial2 = pd.read_csv(
    PROJECT_ROOT / "data/interim/trial2_daily_quality_summary.csv",
    parse_dates=["date"]
)

trial1["trial"] = "Trial 1"
trial2["trial"] = "Trial 2"

daily = pd.concat([trial1, trial2], ignore_index=True)

print("データ件数:", len(daily))
display(daily.head())

データ件数: 209


,date,pen,count,min_mass,median_mass,max_mass,over_10kg,trial
0,2019-06-19,1,546957,-0.43,-0.30,82.15,1166,Trial 1
1,2019-06-19,2,546949,-0.37,-0.22,-0.13,0,Trial 1
2,2019-06-19,3,546539,-0.52,-0.27,-0.05,0,Trial 1
3,2019-06-20,1,789613,-0.47,-0.34,-0.10,0,Trial 1
4,2019-06-20,2,789672,-0.41,-0.25,-0.08,0,Trial 1


In [2]:
# データベースの保存先
db_path = PROJECT_ROOT / "data" / "processed" / "broiler_monitoring.db"

# 保存先フォルダを作成
db_path.parent.mkdir(parents=True, exist_ok=True)

# SQLiteに接続
conn = sqlite3.connect(db_path)

# 日付を文字列に変換して保存
daily_sql = daily.copy()
daily_sql["date"] = daily_sql["date"].dt.strftime("%Y-%m-%d")

# DataFrameをSQLテーブルに保存
daily_sql.to_sql(
    "daily_summary",
    conn,
    if_exists="replace",
    index=False
)

print("データベース作成完了！")
print("保存先:", db_path)

データベース作成完了！
保存先: ..\data\processed\broiler_monitoring.db


In [3]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT trial) AS trial_count,
    COUNT(DISTINCT pen) AS pen_count
FROM daily_summary;
"""

result = pd.read_sql_query(query, conn)

display(result)

,total_rows,trial_count,pen_count
0,209,2,3


In [4]:
query = """
SELECT
    trial,
    pen,
    COUNT(*) AS days,
    ROUND(AVG(median_mass), 2) AS avg_median_mass,
    ROUND(MIN(median_mass), 2) AS min_median_mass,
    ROUND(MAX(median_mass), 2) AS max_median_mass
FROM daily_summary
GROUP BY trial, pen
ORDER BY trial, pen;
"""

group_summary = pd.read_sql_query(query, conn)

display(group_summary)

,trial,pen,days,avg_median_mass,min_median_mass,max_median_mass
0,Trial 1,1,34,51.89,-0.34,106.02
1,Trial 1,2,34,39.78,-0.29,86.07
2,Trial 1,3,34,30.50,-0.50,69.10
3,Trial 2,1,36,30.90,-0.22,79.91
4,Trial 2,2,35,28.58,-0.19,72.93
5,Trial 2,3,36,27.99,-0.11,69.55


In [5]:
query = """
SELECT
    trial,
    pen,
    date,
    median_mass,

    LAG(median_mass) OVER (
        PARTITION BY trial, pen
        ORDER BY date
    ) AS previous_mass,

    ROUND(
        median_mass - LAG(median_mass) OVER (
            PARTITION BY trial, pen
            ORDER BY date
        ),
        2
    ) AS mass_change

FROM daily_summary
ORDER BY trial, pen, date;
"""

daily_changes = pd.read_sql_query(query, conn)

display(daily_changes.head(10))

,trial,pen,date,median_mass,previous_mass,mass_change
0,Trial 1,1,2019-06-19,-0.30,NaN,NaN
1,Trial 1,1,2019-06-20,-0.34,-0.30,-0.04
2,Trial 1,1,2019-06-21,-0.23,-0.34,0.11
3,Trial 1,1,2019-06-22,-0.12,-0.23,0.11
4,Trial 1,1,2019-06-23,0.31,-0.12,0.43
5,Trial 1,1,2019-06-24,0.71,0.31,0.40
6,Trial 1,1,2019-06-25,0.45,0.71,-0.26
7,Trial 1,1,2019-06-26,2.29,0.45,1.84
8,Trial 1,1,2019-06-27,34.67,2.29,32.38
9,Trial 1,1,2019-06-28,10.10,34.67,-24.57


In [6]:
query = """
WITH daily_changes AS (
    SELECT
        trial,
        pen,
        date,
        median_mass,
        LAG(median_mass) OVER (
            PARTITION BY trial, pen
            ORDER BY date
        ) AS previous_mass
    FROM daily_summary
),
changes AS (
    SELECT
        trial,
        pen,
        date,
        median_mass,
        previous_mass,
        ROUND(median_mass - previous_mass, 2) AS mass_change
    FROM daily_changes
)
SELECT
    trial,
    pen,
    date,
    median_mass,
    previous_mass,
    mass_change
FROM changes
WHERE mass_change IS NOT NULL
ORDER BY ABS(mass_change) DESC
LIMIT 10;
"""

top_changes = pd.read_sql_query(query, conn)

display(top_changes)

,trial,pen,date,median_mass,previous_mass,mass_change
0,Trial 1,1,2019-06-27,34.67,2.29,32.38
1,Trial 1,1,2019-06-28,10.10,34.67,-24.57
2,Trial 1,1,2019-07-03,54.55,31.00,23.55
3,Trial 1,1,2019-07-18,77.69,95.02,-17.33
4,Trial 1,3,2019-07-22,69.10,53.36,15.74
5,Trial 1,2,2019-07-01,19.71,5.58,14.13
6,Trial 1,3,2019-07-01,21.84,7.98,13.86
7,Trial 1,1,2019-07-02,31.00,17.75,13.25
8,Trial 1,2,2019-07-18,63.04,76.21,-13.17
9,Trial 1,3,2019-07-13,59.41,47.38,12.03


In [7]:
query = """
SELECT
    trial,
    date,
    COUNT(DISTINCT pen) AS pen_count,
    ROUND(
        MAX(median_mass) - MIN(median_mass),
        3
    ) AS pen_spread
FROM daily_summary
GROUP BY trial, date
ORDER BY trial, date;
"""

sql_pen_spread = pd.read_sql_query(query, conn)

display(sql_pen_spread.head(10))

,trial,date,pen_count,pen_spread
0,Trial 1,2019-06-19,3,0.08
1,Trial 1,2019-06-20,3,0.24
2,Trial 1,2019-06-21,3,0.27
3,Trial 1,2019-06-22,3,0.09
4,Trial 1,2019-06-23,3,0.82
5,Trial 1,2019-06-24,3,0.93
6,Trial 1,2019-06-25,3,1.02
7,Trial 1,2019-06-26,3,1.08
8,Trial 1,2019-06-27,3,28.15
9,Trial 1,2019-06-28,3,0.83


In [8]:
query = """
WITH daily_spread AS (
    SELECT
        trial,
        date,
        COUNT(DISTINCT pen) AS pen_count,
        MAX(median_mass) - MIN(median_mass) AS pen_spread
    FROM daily_summary
    GROUP BY trial, date
),
spread_changes AS (
    SELECT
        trial,
        date,
        pen_count,
        pen_spread,
        LAG(pen_spread) OVER (
            PARTITION BY trial
            ORDER BY date
        ) AS previous_spread
    FROM daily_spread
)
SELECT
    trial,
    date,
    pen_count,
    ROUND(pen_spread, 3) AS pen_spread,
    ROUND(previous_spread, 3) AS previous_spread,
    ROUND(pen_spread - previous_spread, 3) AS spread_change
FROM spread_changes
ORDER BY trial, date;
"""

sql_spread_changes = pd.read_sql_query(query, conn)

display(sql_spread_changes.head(10))

,trial,date,pen_count,pen_spread,previous_spread,spread_change
0,Trial 1,2019-06-19,3,0.08,NaN,NaN
1,Trial 1,2019-06-20,3,0.24,0.08,0.16
2,Trial 1,2019-06-21,3,0.27,0.24,0.03
3,Trial 1,2019-06-22,3,0.09,0.27,-0.18
4,Trial 1,2019-06-23,3,0.82,0.09,0.73
5,Trial 1,2019-06-24,3,0.93,0.82,0.11
6,Trial 1,2019-06-25,3,1.02,0.93,0.09
7,Trial 1,2019-06-26,3,1.08,1.02,0.06
8,Trial 1,2019-06-27,3,28.15,1.08,27.07
9,Trial 1,2019-06-28,3,0.83,28.15,-27.32


In [9]:
# Trialごとに95パーセンタイルを計算
sql_thresholds = (
    sql_spread_changes.groupby("trial")["spread_change"]
    .quantile(0.95)
)

# Trialごとの基準値を付与
sql_spread_changes["threshold"] = (
    sql_spread_changes["trial"].map(sql_thresholds)
)

# 急拡大候補を抽出
sql_candidates = sql_spread_changes[
    sql_spread_changes["spread_change"]
    >= sql_spread_changes["threshold"]
].copy()

display(
    sql_candidates[
        ["trial", "date", "pen_spread", "spread_change", "threshold"]
    ]
)

,trial,date,pen_spread,spread_change,threshold
8,Trial 1,2019-06-27,28.150,27.070,20.302
14,Trial 1,2019-07-03,32.540,25.510,20.302
66,Trial 2,2019-10-11,9.520,4.910,4.427
69,Trial 2,2019-10-14,14.165,6.275,4.427


In [11]:
# 03と同じ処理をpandasで再計算
check = daily.groupby(["trial", "date"])["median_mass"].agg(
    lambda x: x.max() - x.min()
).reset_index(name="python_spread")

# SQL側の日付をdatetime型に変換
sql_spread_changes["date"] = pd.to_datetime(sql_spread_changes["date"])

# SQLの結果と結合
comparison = sql_spread_changes.merge(
    check,
    on=["trial", "date"]
)

# SQLとPythonの計算結果の差
comparison["difference"] = (
    comparison["pen_spread"] - comparison["python_spread"]
).abs()

print("比較した日数:", len(comparison))
print("最大誤差:", comparison["difference"].max())
print("計算結果の一致:", (comparison["difference"] < 0.001).all())

比較した日数: 70
最大誤差: 7.105427357601002e-15
計算結果の一致: True


In [12]:
conn.close()
print("データベース接続を終了しました。")

データベース接続を終了しました。


## 分析結果と考察

### 1. SQLによるデータ集計

Pythonで作成した日次集計データ209件をSQLiteに格納し、
SQLを用いてTrial別・Pen別の集計を実施した。

Trial 1ではPen間の平均値に比較的大きな差がみられた一方、
Trial 2では各Penの平均値が比較的近い傾向にあった。

### 2. 時系列変化の分析

SQLのウィンドウ関数LAG()を使用し、
各Penの日次中央値およびPen間差の変化量を算出した。

変化量の絶対値が大きい上位10件は、すべてTrial 1に集中していた。

また、Trialごとの95パーセンタイルを基準に急拡大候補を抽出した結果、
03のPython分析と同じ4件が検出された。

### 3. PythonとSQLの結果比較

SQLとpandasで算出したPen間差について、70日分を比較した。

- 比較日数：70日
- 最大誤差：約7.11 × 10^-15
- 許容誤差0.001未満での一致：True

この結果から、異なる処理方法でも同等の集計結果が得られることを確認した。

### 4. 考察と今後の課題

SQLiteを使用することで、データの保存・集計・抽出をSQLで実施できた。

また、PythonとSQLの計算結果を比較することで、
処理の整合性を確認できた。

ただし、プラットフォーム荷重の変化は鶏の健康状態を
直接示すものではなく、計測条件などの影響も考えられる。

今後は、SQLによるデータ抽出とPythonによる可視化を組み合わせ、
モニタリング結果を確認しやすい仕組みへの発展を検討する。